# Clinical Lakehouse & Patient AI Copilot

> **Run environment:** Databricks (Unity Catalog, serverless compute) — cells use `%python` / `%sql` Databricks magic.  
> **Local scripts** for DuckDB analytics and the text-to-SQL agent live in `scripts/`.

| | |
|---|---|
| **Data** | 3,455 synthetic patients · Synthea FHIR R4 · ~13 GB JSON bundles |
| **Pipeline** | Auto Loader → Bronze → Silver → Gold → MLflow |
| **ML result** | Random Forest · F1 0.599 · Avg Precision 0.669 · Recall (readmitted) 0.70 |
| **AI layer** | LLM text-to-SQL agent (OpenAI function calling over DuckDB/Parquet) |

```
Synthea → Unity Catalog Volume
  └── Auto Loader ──► bronze_fhir_bundles          (raw JSON, one row per patient file)
         └── PySpark ──► silver_patients            (demographics)
                      ──► silver_encounters         (visit history + encounter class)
                      ──► silver_conditions         (diagnoses)
                      ──► silver_observations       (vitals / labs)
                 └── Window fns ──► gold_patient_features  (1 row/patient + readmission label)
                        └── MLflow ──► readmission_risk_model  (UC Model Registry)
                               └── Parquet export ──► DuckDB ──► text-to-SQL agent
```

## Phase 1: Bronze Layer — Raw Ingestion

**Goal:** Land every FHIR JSON bundle in Delta Lake exactly as received — no parsing, no schema enforcement.  
A permanent fallback if anything downstream breaks.

**Key design choices:**
- `cloudFiles` (Auto Loader) with `binaryFile`: reads files as raw bytes, handles new arrivals without a `LIST` on every run
- `CAST(content AS STRING)` → `raw_json`: preserves the full JSON as a single string column; Silver will parse it
- `trigger(availableNow=True)`: process current Volume contents and stop — right for a one-time historical dump
- Checkpoint path: Auto Loader state persisted separately so re-runs skip already-ingested files


In [ ]:
%python
raw_path = "/Volumes/main/default/synthea_raw/fhir"
checkpoint_path = "/Volumes/main/default/synthea_raw/_checkpoints/bronze_fhir_bundles"
bronze_table = "main.default.bronze_fhir_bundles"

bronze_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "binaryFile")
        .load(raw_path)
        .selectExpr(
            "path as source_file",
            "CAST(content AS STRING) as raw_json",
            "modificationTime as file_modification_time",
            "current_timestamp() as ingestion_time"
        )
)

(bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

## Phase 2: Silver Layer — FHIR Parsing

FHIR R4 bundles are deeply nested, polymorphic JSON. Each file is one patient's complete medical history —  
a single `Bundle` with an `entry` array that mixes all resource types (Patient, Encounter, Condition, Observation, …) in any order.

**Parsing strategy used across all Silver tables:**
1. Infer schema from 20 sample files (not all 13 GB — full inference silently ran out of memory on free-tier serverless)
2. `from_json(raw_json, schema)` → structured `bundle` column
3. `explode(bundle.entry)` → one row per resource, all types mixed
4. Filter on `resourceType`, extract the fields we need
5. Resolve `urn:uuid:<id>` references — Synthea uses this format, not `Patient/<id>`; wrong regex → silent null `patient_id` → broken joins everywhere


### Silver: Patients

One `Patient` resource per bundle. Main data-quality steps: dedup on `patient_id`, drop rows missing `patient_id` or `birth_date` (downstream `age` calculation requires it).


In [ ]:
%python
from pyspark.sql import functions as F

raw_path = "/Volumes/main/default/synthea_raw/fhir"
bronze_df = spark.table("main.default.bronze_fhir_bundles")

# sample schema inference — full 13 GB inference silently fails on free-tier serverless
sample_files = [f.path for f in dbutils.fs.ls(raw_path)[:20]]
bundle_schema = spark.read.option("multiLine", "true").json(sample_files).schema

parsed_df = bronze_df.withColumn("bundle", F.from_json(F.col("raw_json"), bundle_schema))
entries_df = parsed_df.select("source_file", F.explode("bundle.entry").alias("entry"))

patients_df = entries_df.filter(F.col("entry.resource.resourceType") == "Patient").select(
    F.col("entry.resource.id").alias("patient_id"),
    F.col("entry.resource.gender").alias("gender"),
    F.col("entry.resource.birthDate").alias("birth_date"),
    F.expr("entry.resource.name[0].family").alias("last_name"),
    F.expr("entry.resource.name[0].given[0]").alias("first_name"),
    F.expr("entry.resource.address[0].city").alias("city"),
    F.expr("entry.resource.address[0].state").alias("state"),
)

patients_clean = (
    patients_df
    .dropDuplicates(["patient_id"])
    .filter(F.col("patient_id").isNotNull())
    .filter(F.col("birth_date").isNotNull())
)

patients_clean.write.format("delta").mode("overwrite").saveAsTable("main.default.silver_patients")

In [ ]:
%sql
SELECT * FROM main.default.silver_patients LIMIT 10;

### Silver: Encounters

Two engineering problems solved here:

**Problem 1 — `urn:uuid` reference format:** FHIR `subject.reference` comes as `urn:uuid:<patient-id>` in Synthea,  
not the `Patient/<id>` format you'd see in a real EHR. Using the wrong regex (`Patient/(.+)`) silently leaves  
`patient_id` empty and breaks every downstream join — with no error, just missing data.

**Problem 2 — missing `encounter_class`:** Not included in the first pass. Added after the readmission label  
came back at 98.7% positive (see Gold section). The class (`AMB`, `IMP`, `EMER`, …) is required to filter  
to clinically meaningful encounters for the CMS-style readmission definition.


In [ ]:
%python
from pyspark.sql import functions as F

bronze_df = spark.table("main.default.bronze_fhir_bundles")
raw_path = "/Volumes/main/default/synthea_raw/fhir"
sample_files = [f.path for f in dbutils.fs.ls(raw_path)[:20]]
bundle_schema = spark.read.option("multiLine", "true").json(sample_files).schema

parsed_df = bronze_df.withColumn("bundle", F.from_json(F.col("raw_json"), bundle_schema))
entries_df = parsed_df.select("source_file", F.explode("bundle.entry").alias("entry"))

encounters_df = entries_df.filter(F.col("entry.resource.resourceType") == "Encounter").select(
    F.col("entry.resource.id").alias("encounter_id"),
    F.expr("entry.resource.subject.reference").alias("patient_reference"),
    F.expr("entry.resource.reasonCode[0].coding[0].display").alias("reason"),
    F.col("entry.resource.period.start").alias("start_time"),
    F.col("entry.resource.period.end").alias("end_time"),
    F.expr("entry.resource.class.code").alias("encounter_class"),  # AMB / IMP / EMER / HH / VR
)

encounters_clean = (
    encounters_df
    .dropDuplicates(["encounter_id"])
    .filter(F.col("encounter_id").isNotNull())
    .filter(F.col("start_time").isNotNull())
    # Synthea uses urn:uuid:<id> — not Patient/<id>
    .withColumn("patient_id", F.regexp_extract("patient_reference", r"urn:uuid:(.+)", 1))
)

encounters_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("main.default.silver_encounters")

### Silver: Conditions

FHIR `Condition` resources carry diagnosis name (SNOMED CT display text), onset date, and clinical status  
(`active` / `resolved` / `inactive`). The `active_condition_count` Gold feature counts only currently active  
diagnoses as a proxy for chronic disease burden.


In [ ]:
%python
from pyspark.sql import functions as F

bronze_df = spark.table("main.default.bronze_fhir_bundles")
raw_path = "/Volumes/main/default/synthea_raw/fhir"
sample_files = [f.path for f in dbutils.fs.ls(raw_path)[:20]]
bundle_schema = spark.read.option("multiLine", "true").json(sample_files).schema

parsed_df = bronze_df.withColumn("bundle", F.from_json(F.col("raw_json"), bundle_schema))
entries_df = parsed_df.select("source_file", F.explode("bundle.entry").alias("entry"))

conditions_df = entries_df.filter(F.col("entry.resource.resourceType") == "Condition").select(
    F.col("entry.resource.id").alias("condition_id"),
    F.expr("entry.resource.subject.reference").alias("patient_reference"),
    F.expr("entry.resource.code.coding[0].display").alias("condition_name"),
    F.expr("entry.resource.code.coding[0].code").alias("condition_code"),
    F.col("entry.resource.onsetDateTime").alias("onset_date"),
    F.expr("entry.resource.clinicalStatus.coding[0].code").alias("clinical_status"),
)

conditions_clean = (
    conditions_df
    .dropDuplicates(["condition_id"])
    .filter(F.col("condition_id").isNotNull())
    .filter(F.col("condition_name").isNotNull())
    .withColumn("patient_id", F.regexp_extract("patient_reference", r"urn:uuid:(.+)", 1))
)

conditions_clean.write.format("delta").mode("overwrite").saveAsTable("main.default.silver_conditions")

In [ ]:
%sql
SELECT condition_name, clinical_status, onset_date, patient_id FROM main.default.silver_conditions LIMIT 10;

### Silver: Observations

`Observation.value` is polymorphic across FHIR resource types — numeric vitals (BMI, heart rate) use `valueQuantity.value`,  
while coded answers like "Tobacco smoking status" have no numeric value at all.  
A hard `CAST` would throw on those rows.

**Fix:** `try_cast(... AS DOUBLE)` returns `null` for non-numeric observations instead of failing.  
Non-numeric rows are still stored; Gold simply gets `null` for their `value_numeric`.


In [ ]:
%python
from pyspark.sql import functions as F

bronze_df = spark.table("main.default.bronze_fhir_bundles")
raw_path = "/Volumes/main/default/synthea_raw/fhir"
sample_files = [f.path for f in dbutils.fs.ls(raw_path)[:20]]
bundle_schema = spark.read.option("multiLine", "true").json(sample_files).schema

parsed_df = bronze_df.withColumn("bundle", F.from_json(F.col("raw_json"), bundle_schema))
entries_df = parsed_df.select("source_file", F.explode("bundle.entry").alias("entry"))

observations_df = entries_df.filter(F.col("entry.resource.resourceType") == "Observation").select(
    F.col("entry.resource.id").alias("observation_id"),
    F.expr("entry.resource.subject.reference").alias("patient_reference"),
    F.expr("entry.resource.code.coding[0].display").alias("observation_name"),
    # try_cast handles polymorphic value field — coded observations (e.g. smoking status) have no numeric value
    F.expr("try_cast(entry.resource.valueQuantity.value AS DOUBLE)").alias("value_numeric"),
    F.expr("entry.resource.valueQuantity.unit").alias("unit"),
    F.col("entry.resource.effectiveDateTime").alias("observed_date"),
)

observations_clean = (
    observations_df
    .dropDuplicates(["observation_id"])
    .filter(F.col("observation_id").isNotNull())
    .filter(F.col("observation_name").isNotNull())
    .withColumn("patient_id", F.regexp_extract("patient_reference", r"urn:uuid:(.+)", 1))
)

observations_clean.write.format("delta").mode("overwrite").saveAsTable("main.default.silver_observations")

In [ ]:
%sql
SELECT observation_name, value_numeric, unit, observed_date, patient_id FROM main.default.silver_observations LIMIT 10;

## Phase 3: Gold Layer — Feature Engineering

One row per patient summarizing their entire history — the ML-ready feature table.

| Column | How |
|---|---|
| `age` | `datediff(today, birth_date) / 365.25` |
| `total_encounters` | `COUNT(*)` grouped by patient |
| `last_encounter_date` | `MAX(start_time)` |
| `active_condition_count` | `countDistinct(condition_name)` filtered to `clinical_status = 'active'` |
| `latest_bmi` | Window: `row_number()` partitioned by `(patient_id, observation_name)` ordered by date desc, keep rank 1 |
| `latest_heart_rate` | Same window pattern |

Window functions (`row_number()`) are the cleanest way to pick the most-recent record per group  
without a correlated subquery per patient.


In [ ]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

patients    = spark.table("main.default.silver_patients")
encounters  = spark.table("main.default.silver_encounters")
conditions  = spark.table("main.default.silver_conditions")
observations = spark.table("main.default.silver_observations")

# encounter history
encounter_features = encounters.groupBy("patient_id").agg(
    F.count("*").alias("total_encounters"),
    F.max("start_time").alias("last_encounter_date"),
)

# chronic condition burden
condition_features = (
    conditions
    .filter(F.col("clinical_status") == "active")
    .groupBy("patient_id")
    .agg(F.countDistinct("condition_name").alias("active_condition_count"))
)

# latest vitals: row_number() keeps the most recent reading per (patient, observation type)
obs_window = Window.partitionBy("patient_id", "observation_name").orderBy(F.col("observed_date").desc())
latest_obs = (
    observations
    .withColumn("rn", F.row_number().over(obs_window))
    .filter(F.col("rn") == 1)
)

latest_bmi        = latest_obs.filter(F.col("observation_name") == "Body mass index (BMI) [Ratio]") \
    .select("patient_id", F.col("value_numeric").alias("latest_bmi"))
latest_heart_rate = latest_obs.filter(F.col("observation_name") == "Heart rate") \
    .select("patient_id", F.col("value_numeric").alias("latest_heart_rate"))

patient_features = (
    patients
    .withColumn("age", F.floor(F.datediff(F.current_date(), F.col("birth_date")) / 365.25))
    .join(encounter_features,  "patient_id", "left")
    .join(condition_features,  "patient_id", "left")
    .join(latest_bmi,          "patient_id", "left")
    .join(latest_heart_rate,   "patient_id", "left")
    .fillna({"total_encounters": 0, "active_condition_count": 0})
    .select("patient_id", "age", "gender", "total_encounters", "last_encounter_date",
            "active_condition_count", "latest_bmi", "latest_heart_rate")
)

patient_features.write.format("delta").mode("overwrite").saveAsTable("main.default.gold_patient_features")

In [ ]:
%sql
SELECT * FROM main.default.gold_patient_features ORDER BY total_encounters DESC LIMIT 10;

### Readmission Label — v1: The Bug

**Definition attempted:** any encounter followed by another encounter within 30 days → `readmitted_30d = 1`.

Built with `lead()` over a window ordered by `start_time` — standard pattern for detecting consecutive events.


In [ ]:
%python
# v1 — broken; kept here to show the diagnostic path
from pyspark.sql import functions as F
from pyspark.sql.window import Window

encounters = spark.table("main.default.silver_encounters")
enc_window = Window.partitionBy("patient_id").orderBy("start_time")

encounters_with_lead = encounters.withColumn(
    "next_encounter_start", F.lead("start_time").over(enc_window)
)

labeled = (
    encounters_with_lead
    .withColumn("days_to_next", F.datediff(F.col("next_encounter_start"), F.col("end_time")))
    .withColumn("readmitted_30d", F.when(F.col("days_to_next").between(0, 30), 1).otherwise(0))
)

patient_label = labeled.groupBy("patient_id").agg(F.max("readmitted_30d").alias("readmitted_30d"))

gold = spark.table("main.default.gold_patient_features")
training_data = gold.join(patient_label, "patient_id", "left").fillna({"readmitted_30d": 0})
training_data.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("main.default.gold_patient_features")

In [ ]:
%sql
-- Expected: should be ~15-20% positive. If near 100%, the label is wrong.
SELECT readmitted_30d, COUNT(*) as patient_count FROM main.default.gold_patient_features GROUP BY readmitted_30d;

readmitted_30d  patient_count
0               45
1               3410

→ 98.7% flagged positive — obviously wrong.


### Diagnosing and Fixing the Readmission Label

**Root cause:** Synthea simulates 70+ years of medical history per patient.  
A typical patient has 700+ encounters over their lifetime — mostly routine ambulatory checkups every few months.  
With that density, *any two consecutive encounters* land within 30 days of each other far more often than not.  
The v1 label measured **"does this patient go to the doctor a lot?"** — not actual hospital readmission risk.

**Real-world definition (CMS):** 30-day hospital readmission = an **inpatient (IMP)** or **emergency (EMER)** admission  
within 30 days of a prior inpatient or emergency *discharge*. Routine office visits don't count.

**Fix:** Filter to `encounter_class IN ('IMP', 'EMER')` before computing the lead window.  
First, verify the actual class codes in the data:


In [ ]:
%sql
SELECT encounter_class, COUNT(*) as count FROM main.default.silver_encounters GROUP BY encounter_class ORDER BY count DESC;

encounter_class  count
AMB              2204687
EMER             89451
IMP              44203
HH               12890
VR               8102


IMP (inpatient) and EMER (emergency) are a small fraction of all encounters — exactly the signal we want to isolate.


In [ ]:
%python
# v2 — fixed: only inpatient and emergency encounters count toward readmission
from pyspark.sql import functions as F
from pyspark.sql.window import Window

encounters = spark.table("main.default.silver_encounters")

# CMS-style: only hospital-level encounters define readmission
significant_encounters = encounters.filter(F.col("encounter_class").isin("IMP", "EMER"))

enc_window = Window.partitionBy("patient_id").orderBy("start_time")

encounters_with_lead = significant_encounters.withColumn(
    "next_encounter_start", F.lead("start_time").over(enc_window)
)

labeled = (
    encounters_with_lead
    .withColumn("days_to_next", F.datediff(F.col("next_encounter_start"), F.col("end_time")))
    .withColumn("readmitted_30d", F.when(F.col("days_to_next").between(0, 30), 1).otherwise(0))
)

patient_label = labeled.groupBy("patient_id").agg(F.max("readmitted_30d").alias("readmitted_30d"))

gold = spark.table("main.default.gold_patient_features").drop("readmitted_30d")
training_data = gold.join(patient_label, "patient_id", "left").fillna({"readmitted_30d": 0})
training_data.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("main.default.gold_patient_features")

In [ ]:
%sql
SELECT readmitted_30d, COUNT(*) as patient_count FROM main.default.gold_patient_features GROUP BY readmitted_30d;

readmitted_30d  patient_count
0               2746
1               709

→ 20.5% positive — matches real-world 15-20% hospital readmission rates (CMS benchmark).


## Phase 4: ML with MLflow

**Task:** Binary classification — predict 30-day hospital readmission risk from the Gold feature table.

**Feature set:** age, total_encounters, active_condition_count, latest_bmi, latest_heart_rate, gender (one-hot)

**Key decisions:**
- `class_weight="balanced"`: the 80/20 split would let a naive model score 79% accuracy by always predicting "no"; balanced weights penalize minority-class misclassification proportionally
- Median imputation for BMI / heart rate: ~15% of Synthea patients have no recorded vitals; dropping them biases training toward patients with heavier clinical contact
- `input_example` required for Unity Catalog model registration (MLflow needs it to infer the model signature)
- Model registered in UC Model Registry (`main.default.readmission_risk_model`) for versioning and governance

**Results:** F1 0.599 · Average Precision 0.669 · Recall (readmitted class) 0.70  
Solid for a genuinely hard problem — ensemble models on real EHR data typically land in the 0.65–0.75 AUC range.


In [ ]:
%python
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, average_precision_score, classification_report

mlflow.set_registry_uri("databricks-uc")

df = spark.table("main.default.gold_patient_features").toPandas()

feature_cols = ["age", "total_encounters", "active_condition_count", "latest_bmi", "latest_heart_rate"]
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())
df["gender_male"] = (df["gender"] == "male").astype(int)
feature_cols.append("gender_male")

X = df[feature_cols]
y = df["readmitted_30d"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

with mlflow.start_run(run_name="readmission_random_forest") as run:
    params = {"n_estimators": 200, "max_depth": 6, "class_weight": "balanced", "random_state": 42}
    mlflow.log_params(params)

    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)

    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    f1            = f1_score(y_test, y_pred)
    avg_precision = average_precision_score(y_test, y_proba)

    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("average_precision", avg_precision)

    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="main.default.readmission_risk_model",
        input_example=X_train.head(5),
    )

    print(f"F1: {f1:.3f}  |  Average Precision: {avg_precision:.3f}")
    print(classification_report(y_test, y_pred))


F1: 0.599  |  Average Precision: 0.669
              precision    recall  f1-score   support

           0       0.92      0.83      0.87       549
           1       0.52      0.70      0.60       142

    accuracy                           0.81       691
   macro avg       0.72      0.77      0.74       691
weighted avg       0.83      0.81      0.82       691


## Phase 5: Export — Parquet + Local Analytics

Gold and Silver tables exported to Parquet in the Volume, then downloaded locally  
(see `scripts/01_setup_synthea.sh` for the `databricks fs cp` commands).

**Local analytics with DuckDB** (`scripts/02_duckdb_analytics.py`) found:
- Readmitted patients average **25 years older** than non-readmitted
- Readmitted patients average **2.7× more encounters** — consistent with known readmission drivers

**Text-to-SQL agent** (`scripts/03_text_to_sql_agent.py`): LLM (GPT-4o-mini via OpenAI function calling)  
writes its own SQL against the DuckDB/Parquet tables to answer free-form clinical questions.  
Chosen over embedding-based RAG because the data is structured, not narrative text.


In [ ]:
%python
# Export Gold table to Parquet for local DuckDB analytics
spark.table("main.default.gold_patient_features").write.mode("overwrite").parquet(
    "/Volumes/main/default/synthea_raw/exports/gold_patient_features_parquet"
)

# Also export Silver tables used by the text-to-SQL agent
spark.table("main.default.silver_conditions").write.mode("overwrite").parquet(
    "/Volumes/main/default/synthea_raw/exports/silver_conditions_parquet"
)
spark.table("main.default.silver_encounters").write.mode("overwrite").parquet(
    "/Volumes/main/default/synthea_raw/exports/silver_encounters_parquet"
)

In [ ]:
%sql
-- Final verification: one row per patient with all features + readmission label
SELECT patient_id, age, gender, total_encounters, active_condition_count,
       latest_bmi, latest_heart_rate, readmitted_30d
FROM main.default.gold_patient_features
LIMIT 10;

## Analytics: Power BI Live Connection

The Gold and Silver Delta tables are queryable live from Power BI Desktop via the  
**Databricks SQL Warehouse connector** — no Parquet export needed for BI.

### Connection steps
1. In Power BI Desktop → **Get Data → Databricks**
2. Set **Server hostname** to your workspace SQL endpoint (e.g. `dbc-d9f2ffec-6664.cloud.databricks.com`)
3. Set **HTTP path** from the SQL Warehouse connection details (Settings → Connection details)
4. Authenticate with a **Personal Access Token** or Azure AD
5. Browse to `main.default` catalog → select tables

### Suggested dashboard pages
| Page | Key visuals |
|---|---|
| **Population Overview** | Patient count by state (map), age distribution (histogram), gender split |
| **Readmission Risk** | % readmitted by age band (bar), avg encounters for readmitted vs not (grouped bar) |
| **Condition Burden** | Top 10 active conditions (bar), condition count vs readmission rate (scatter) |
| **Encounter Trends** | Encounter class breakdown (donut), encounters over time (line) |

### Key DAX measures
```dax
Readmission Rate =
    DIVIDE(
        CALCULATE(COUNTROWS(gold_patient_features), gold_patient_features[readmitted_30d] = 1),
        COUNTROWS(gold_patient_features)
    )

Avg Age Readmitted =
    CALCULATE(AVERAGE(gold_patient_features[age]), gold_patient_features[readmitted_30d] = 1)

Avg Encounters Readmitted =
    CALCULATE(AVERAGE(gold_patient_features[total_encounters]), gold_patient_features[readmitted_30d] = 1)
```

> **DirectQuery mode** (not Import) keeps the data live against the Delta tables.  
> The SQL Warehouse auto-suspends when idle — no compute cost while Power BI is closed.
